In [10]:
from dataclasses import dataclass
from datetime import date, timedelta,datetime

@dataclass
class VoucherDateRange:
    start_date:str
    end_date:str
    archive_folder: str
    export_filename: str

    def __str__(self):
        return (
            f"凭证时间范围:\n"
            f"  开始日期: {self.start_date}\n"
            f"  结束日期: {self.end_date}\n"
            f"  归档目录: {self.archive_folder}\n"
            f"  导出文件: {self.export_filename}"
        )


    
def get_voucher_date_range(current_datetime:datetime, offset_day: int = 3)->VoucherDateRange:
    """
    规则：
    1. 如果当前日期的 day <= offset_day：
       查询：上个月1号 至 昨天

    2. 如果当前日期的 day > offset_day：
       查询：本月1号 至 昨天

    """
    if current_datetime is None:
        current_datetime = datetime.now()
    today = current_datetime.date()
    
    if today.day <= offset_day:
        # 本月第一天
        current_month_first_day = today.replace(day=1)

        # 上个月最后一天
        end_date  = current_month_first_day - timedelta(days=1)

        # 上个月第一天
        start_date = end_date .replace(day=1)

    else:
        # 本月第一天
        start_date = today.replace(day=1)
        # 昨天
        end_date = today - timedelta(days=1)


    output_start_date = start_date.strftime("%Y-%m-%d")
    output_end_date = end_date.strftime("%Y-%m-%d")
    #格式化日期
    yesterday = today - timedelta(days=1)
    yesterday_yyMMdd = yesterday.strftime("%y%m%d")
    archive_forlder = f"{end_date.strftime('%y')}年{end_date.month}月"
    export_filename = f"{yesterday_yyMMdd}-SAP已付款数据-{archive_forlder}.xlsx"

    return VoucherDateRange(
        start_date=output_start_date,
        end_date=output_end_date,
        archive_folder=archive_forlder,
        export_filename=export_filename)

dt = datetime(2026, 4, 30, 10, 30, 0)
rt = get_voucher_date_range(dt)
print(dt,rt)

dt = datetime(2026, 5, 1, 10, 30, 0)
rt = get_voucher_date_range(dt)
print(dt,rt)

dt = datetime(2026, 5, 2, 10, 30, 0)
rt = get_voucher_date_range(dt)
print(dt,rt)

dt = datetime(2026, 5, 3, 10, 30, 0)
rt = get_voucher_date_range(dt)
print(dt,rt)

dt = datetime(2026, 5, 4, 10, 30, 0)
rt = get_voucher_date_range(dt)
print(dt,rt)


2026-04-30 10:30:00 凭证时间范围:
  开始日期: 2026-04-01
  结束日期: 2026-04-29
  归档目录: 26年4月
  导出文件: 260429-SAP已付款数据-26年4月.xlsx
2026-05-01 10:30:00 凭证时间范围:
  开始日期: 2026-04-01
  结束日期: 2026-04-30
  归档目录: 26年4月
  导出文件: 260430-SAP已付款数据-26年4月.xlsx
2026-05-02 10:30:00 凭证时间范围:
  开始日期: 2026-04-01
  结束日期: 2026-04-30
  归档目录: 26年4月
  导出文件: 260501-SAP已付款数据-26年4月.xlsx
2026-05-03 10:30:00 凭证时间范围:
  开始日期: 2026-04-01
  结束日期: 2026-04-30
  归档目录: 26年4月
  导出文件: 260502-SAP已付款数据-26年4月.xlsx
2026-05-04 10:30:00 凭证时间范围:
  开始日期: 2026-05-01
  结束日期: 2026-05-03
  归档目录: 26年5月
  导出文件: 260503-SAP已付款数据-26年5月.xlsx


In [ ]:
import hashlib
import base64
import hmac
import json

def gen_bot_msg(secret:str,msg:str="SAP 付款数据导出成功"):
    timestamp = datetime.now().timestamp()
    # 拼接timestamp和secret
    string_to_sign = '{}\n{}'.format(timestamp, secret)
    hmac_code = hmac.new(string_to_sign.encode("utf-8"), digestmod=hashlib.sha256).digest()

    # 对结果进行base64处理
    sign = base64.b64encode(hmac_code).decode('utf-8')
 
    body = {"timestamp":timestamp,"sign":sign,"msg_type":"text","content":{"text":msg}}
    
    return json.dumps(body)

t = gen_bot_msg("d8iVK6TxqVoN0HxJvzHsqb")
print(t)


{"timestamp": 1777734101.905372, "sign": "ZtiFdVKy9OYBIs4VkhfnIQQ55XVhAnrkp6WaGiP1tYE=", "msg_type": "text", "content": {"text": "SAP \u4ed8\u6b3e\u6570\u636e\u5bfc\u51fa\u6210\u529f"}}


In [ ]:
# 构造函数接收装饰器函数

class LogDecorator:
    def __init__(self, func):
        # func 是被装饰的原函数
        self.func = func

    def __call__(self, *args, **kwargs):
        print(f"开始执行：{self.func.__name__}")

        result = self.func(*args, **kwargs)

        print(f"执行结束：{self.func.__name__}")

        return result


@LogDecorator
def say_hello(name: str):
    print(f"你好，{name}")


say_hello("张三")

"""
1. @LogDecorator 等价：LogDecorator(say_hello) 可调用的对象
2. LogDecorator(say_hello)(name) 等价：LogDecorator可调用对象.__call__(name)
"""


开始执行：say_hello
你好，张三
执行结束：say_hello


In [ ]:
# __call__ 接收函数

class LogDecorator:
    def __init__(self, prefix: str):
        # prefix 是装饰器自己的配置参数
        self.prefix = prefix

    def __call__(self, func):
        # func 是被装饰的原函数
        def wrapper(*args, **kwargs):
            print(f"[{self.prefix}] 开始执行：{func.__name__}")

            result = func(*args, **kwargs)

            print(f"[{self.prefix}] 执行结束：{func.__name__}")

            return result

        return wrapper


@LogDecorator(prefix="用户模块")
def create_user(username: str):
    print(f"创建用户：{username}")


create_user("张三")
""" 
func = LogDecorator("配置参数")(func)

1. 第1步：decorator = LogDecorator(prefix="用户模块") 执行 __init__()
2. 第2步：再装饰函数 create_user = decorator(create_user) 执行 __call__(func)
3. 
"""


[用户模块] 开始执行：create_user
创建用户：张三
[用户模块] 执行结束：create_user


In [1]:
# 带参数的类装饰器
class ArgsDecorator:
    def __init__(self,name:str):
        self.name = name
    
    def __call__(self,func):
        def wrapper(*args,**kwargs):
            print(f"{self.name}：执行前")
            result = func(*args, **kwargs)
            print(f"{self.name}：执行后")
            return result
        return wrapper

@ArgsDecorator("日志装饰器")
def demo():
    print("demo 函数执行")

demo()


日志装饰器：执行前
demo 函数执行
日志装饰器：执行后


In [1]:
# 企业项目案例：接口权限校验
# 问题
# 权限逻辑写死在业务函数里
# 多个函数都要重复判断
# 不方便复用
# 不方便统一修改权限规则
from functools import wraps

class RequireRole:
    def __init__(self, role: str):
        self.role = role

    def __call__(self, func):
        @wraps(func)
        def wrapper(user_role: str, *args, **kwargs):
            if user_role != self.role:
                raise PermissionError(f"需要 {self.role} 权限")

            return func(user_role, *args, **kwargs)

        return wrapper

@RequireRole("admin")
def delete_user(user_role: str, user_id: int):
    print(f"删除用户：{user_id}")


delete_user("admin", 1001)


删除用户：1001


In [ ]:
# 日志装饰器
class LogDecorator:
    def __init__(self, prefix: str):
        # prefix 是装饰器自己的配置参数
        self.prefix = prefix
    def __call__(self, func):
        # func 是被装饰的原函数
        def wrapper(*args, **kwargs):
            print(f"[{self.prefix}] 开始执行：{func.__name__}")

            result = func(*args, **kwargs)

            print(f"[{self.prefix}] 执行结束：{func.__name__}")

            return result

        return wrapper


@LogDecorator(prefix="用户模块")
def create_user(username: str):
    print(f"创建用户：{username}")


create_user("张三")


[用户模块] 开始执行：create_user
创建用户：张三
[用户模块] 执行结束：create_user


## 嵌套装饰器语法
看到：
```python
@A
@B
@C
def func():
    pass
```
还原：
```python
func = A(B(C(func)))

func()
```
结论：
1. 谁先装饰？离函数最近的装饰器先执行
2. 调用时谁先执行？最外层先进入,最内层最后进入【洋葱模型】


### 带着问题学习：
1. 类装饰器嵌套到底解决什么问题？
场景：企业项目中，一个函数通常不止一种功能。
例如: 创建用户

可能同时需要：
- 日志
- 权限
- 缓存
- 限流
- 重试
- 事务
- 性能统计

如果全部写进函数：代码会越来越乱。
```python
def create_order():
    check_permission()
    write_log()
    start_transaction()
    ...
```

所以企业里会使用：
```python
@Log
@Permission
@Transaction
def create_order():
    ...
```

分析它的真实问题：
核心业务与非核心业务的解耦
所以这种场景适合：装饰器嵌套。

### 为什么企业喜欢这种结构？
```python
@Retry
@Cache
@Permission
@Transaction
@Log
def create_order():
    ...
```

优点：
- 功能解耦
- 横切逻辑复用
- AOP 
- 中间件链
- 插件机制

中间件责任链

## 企业里的真实执行链
请求
 ↓
日志中间件
 ↓
Trace 中间件
 ↓
权限中间件
 ↓
事务中间件
 ↓
限流中间件
 ↓
业务函数

对应
```python
@Log
@Trace
@Permission
@Transaction
@RateLimit
def api():
    ...
```

In [1]:
class A:
    def __init__(self, func):
        print("A.__init__")
        self.func = func

    def __call__(self, *args, **kwargs):
        print("A 开始")

        result = self.func(*args, **kwargs)

        print("A 结束")

        return result


class B:
    def __init__(self, func):
        print("B.__init__")
        self.func = func

    def __call__(self, *args, **kwargs):
        print("B 开始")

        result = self.func(*args, **kwargs)

        print("B 结束")

        return result


class C:
    def __init__(self, func):
        print("C.__init__")
        self.func = func

    def __call__(self, *args, **kwargs):
        print("C 开始")

        result = self.func(*args, **kwargs)

        print("C 结束")

        return result

@A
@B
@C
def hello():
    print("hello 函数")


C.__init__
B.__init__
A.__init__


In [ ]:
# 案例：
class Log:
    def __init__(self, module: str):
        self.module = module

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            print(f"[日志-{self.module}] 开始")

            result = func(*args, **kwargs)

            print(f"[日志-{self.module}] 结束")

            return result

        return wrapper


class Permission:
    def __init__(self, role: str):
        self.role = role

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            print(f"[权限检查] 需要角色：{self.role}")

            return func(*args, **kwargs)

        return wrapper


@Log("订单模块")
@Permission("admin")
def create_order():
    print("创建订单")
create_order()




[日志-订单模块] 开始
[权限检查] 需要角色：admin
创建订单
[日志-订单模块] 结束


In [ ]:
# 为什么企业里更常用 __call__ 接收函数？
# 因为企业项目里的装饰器多数都需要配置。

@retry(times=3)
def request_api():
    ...

@permission("user:delete")
def delete_user():
    ...

@rate_limit(limit=100)
def query_data():
    ...

@cache(ttl=300)
def get_user_info():
    ...

# 因此通常会设计：
class Decorator:
    def __init__(self, 配置参数):
        ...

    def __call__(self, func):
        ...


## 总结
### 类装饰器-构造函数方式
适合：装饰器不需要参数

核心结构：

```python
class Decorator:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

@Decorator
def hi():pass
```

### 类装饰器-__call__ 方式
适合：装饰器需要参数
核心结构：
```python
class Decorator:
    def __init__(self, config):
        self.config = config

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)

        return wrapper
```

## 函数装饰器三层嵌套原理
函数装饰器三层嵌套原理
functools.wraps
AOP（面向切面编程）
FastAPI 中间件
Starlette Middleware
责任链模式
依赖注入
Descriptor（描述符）